In [ ]:
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

# ── Linear regression from scratch ────────────────────────────────────

# Toy dataset: y = 2x + 1  plus some noise
torch.manual_seed(0)
X = torch.randn(100, 1)                    # 100 examples, 1 feature
y = 2 * X + 1 + 0.2 * torch.randn(100, 1) # true relationship + noise

# Model: a single linear layer (no bias separately; nn.Linear includes it)
model = nn.Linear(in_features=1, out_features=1)

# Loss and optimiser
loss_fn   = nn.MSELoss()                          # L2 / mean squared error
optimiser = torch.optim.SGD(model.parameters(), lr=0.1)

# Training loop
for epoch in range(200):
    optimiser.zero_grad()           # 1. clear gradients from last step
    y_pred = model(X)               # 2. forward pass: compute predictions
    loss   = loss_fn(y_pred, y)     # 3. compute scalar loss
    loss.backward()                 # 4. backward pass: compute gradients
    optimiser.step()                # 5. update weights

    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}  Loss: {loss.item():.4f}')

# Inspect learned weights — should be close to w=2, b=1
w, b = model.weight.item(), model.bias.item()
print(f'Learned: y = {w:.3f}·x + {b:.3f}')   # ≈ y = 2.0·x + 1.0

plt.plot(X.numpy(), y.numpy(), 'o', label='data')
plt.plot(X.numpy(), model(X).detach().numpy(), label='model')
plt.legend()
plt.show()


In [ ]:
import torch
from matplotlib import pyplot as plt

# ── Polynomial regression ───────────────────────

# The model is still LINEAR IN THE WEIGHTS — just the features are non-linear.

def poly_features(x, degree):
    return torch.cat([x ** d for d in range(degree + 1)], dim=1)

torch.manual_seed(1)

x_train = torch.linspace(-1, 1, 20).unsqueeze(1)
y_train = torch.sin(3 * x_train) + 0.1 * torch.randn_like(x_train)

# Dense grid for plotting
x_plot = torch.linspace(-1, 1, 500).unsqueeze(1)

plt.plot(
    x_train.numpy(),
    y_train.numpy(),
    'o',
    label='training data'
)

# Show the true underlying function
plt.plot(
    x_plot.numpy(),
    torch.sin(x_plot).numpy(),
    '--',
    label='true function'
)

for degree in [1, 3, 15]:

    X_train = poly_features(x_train, degree)
    X_plot  = poly_features(x_plot, degree)

    # Exact least-squares solution
    w = torch.linalg.lstsq(X_train, y_train).solution

    y_train_pred = X_train @ w
    y_plot_pred  = X_plot @ w

    loss = torch.mean((y_train_pred - y_train) ** 2)

    print(
        f'Degree {degree:2d}  '
        f'train loss: {loss.item():.6f}'
    )

    plt.plot(
        x_plot.numpy(),
        y_plot_pred.numpy(),
        label=f'degree {degree}'
    )

plt.ylim(-1.5, 1.5)
plt.legend()
plt.show()

# Degree  1  → high loss (underfitting)
# Degree  3  → low loss (good fit)
# Degree 15  → near-zero loss (overfitting — memorised noise)

In [ ]:
import torch
from torch.utils.data import TensorDataset, random_split
from matplotlib import pyplot as plt

# ── Polynomial features ────────────────────────────────────────────────

def poly_features(x, degree):
    """Expand scalar x into [x^0, x^1, ..., x^degree]."""
    return torch.cat([x ** d for d in range(degree + 1)], dim=1)

def fit_polynomial(x, y, degree):
    """Fit polynomial coefficients using least squares."""
    X = poly_features(x, degree)
    return torch.linalg.lstsq(X, y).solution

def predict_polynomial(x, w):
    degree = len(w) - 1
    X = poly_features(x, degree)
    return X @ w

def mse(y_pred, y):
    return ((y_pred - y) ** 2).mean().item()

# ── Generate data ─────────────────────────────────────────────────────

torch.manual_seed(42)

N = 60

# x in [-1, 1] keeps polynomial features numerically well behaved
x_all = 2 * torch.rand(N, 1) - 1

# Underlying function + noise
y_all = torch.sin(3 * x_all) + 0.2 * torch.randn_like(x_all)

# ── Standard train / validation / test split ──────────────────────────

# 60 / 20 / 20 split
n_train = int(0.6 * N)
n_val   = int(0.2 * N)
n_test  = N - n_train - n_val

dataset = TensorDataset(x_all, y_all)

train_ds, val_ds, test_ds = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(1)
)

# Convert subsets back into tensors
def subset_to_tensors(ds):
    x = torch.stack([sample[0] for sample in ds])
    y = torch.stack([sample[1] for sample in ds])
    return x, y

x_train, y_train = subset_to_tensors(train_ds)
x_val,   y_val   = subset_to_tensors(val_ds)
x_test,  y_test  = subset_to_tensors(test_ds)

print(
    f'Train: {len(train_ds)}  '
    f'Val: {len(val_ds)}  '
    f'Test: {len(test_ds)}'
)

# ── Hyperparameter tuning ─────────────────────────────────────────────

# Polynomial degree is the hyperparameter
degrees = range(1, 16)

train_losses = []
val_losses   = []

for degree in degrees:

    # Fit parameters ONLY using training data
    w = fit_polynomial(x_train, y_train, degree)

    # Evaluate on training data
    train_pred = predict_polynomial(x_train, w)
    train_loss = mse(train_pred, y_train)

    # Evaluate on validation data
    val_pred = predict_polynomial(x_val, w)
    val_loss = mse(val_pred, y_val)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f'Degree {degree:2d}: '
        f'train loss = {train_loss:.4f}, '
        f'val loss = {val_loss:.4f}'
    )

# Choose hyperparameter using validation set
best_degree = degrees[
    torch.tensor(val_losses).argmin().item()
]

print(f'\nBest polynomial degree: {best_degree}')

# ── Final evaluation on test set ──────────────────────────────────────

# Refit using the selected degree
w_best = fit_polynomial(x_train, y_train, best_degree)

test_pred = predict_polynomial(x_test, w_best)
test_loss = mse(test_pred, y_test)

print(f'Test loss: {test_loss:.4f}')

# ── Plot validation curve ─────────────────────────────────────────────

plt.figure()
plt.plot(degrees, train_losses, 'o-', label='train')
plt.plot(degrees, val_losses,   'o-', label='validation')
plt.xlabel('Polynomial degree')
plt.ylabel('MSE')
plt.legend()
plt.show()

# ── Plot selected model ───────────────────────────────────────────────

x_plot = torch.linspace(-1, 1, 500).unsqueeze(1)
y_plot = predict_polynomial(x_plot, w_best)

plt.figure()
plt.plot(x_plot, torch.sin(3 * x_plot), '--', label='true function')
plt.plot(x_plot, y_plot, label=f'degree {best_degree}')
plt.plot(x_train, y_train, 'o', label='train')
plt.plot(x_val,   y_val,   'x', label='validation')
plt.plot(x_test,  y_test,  '+', label='test')
plt.legend()
plt.show()

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

# ── Standard train/val/test split ─────────────────────────────────────
torch.manual_seed(42)
N = 1000
X_all = torch.randn(N, 10)
y_all = torch.randint(0, 2, (N,))

# 70 / 15 / 15 split
n_train, n_val = int(0.7 * N), int(0.15 * N)
n_test  = N - n_train - n_val
dataset = TensorDataset(X_all, y_all)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
test_loader  = DataLoader(test_ds,  batch_size=64)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

# ── Validation loop (no gradient computation) ─────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()                    # disables dropout, batch-norm update
    total_loss, correct = 0, 0
    with torch.no_grad():           # no gradient tracking needed
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            total_loss += loss_fn(logits, y_batch).item()
            correct    += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)


In [ ]:
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

# ── Binary logistic regression on synthetic data ──────────────────────
torch.manual_seed(0)
N = 200
# Class 0: centred at (-1, -1);  Class 1: centred at (1, 1)
X0 = torch.randn(N // 2, 2) - 1
X1 = torch.randn(N // 2, 2) + 1
X  = torch.cat([X0, X1])
y  = torch.cat([torch.zeros(N // 2), torch.ones(N // 2)]).long()

# nn.Linear gives us w^T x + b
# nn.BCEWithLogitsLoss = sigmoid + binary cross-entropy, numerically stable
model     = nn.Linear(2, 1)               # 2 input features, 1 output logit
loss_fn   = nn.BCEWithLogitsLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(300):
    optimiser.zero_grad()
    logits = model(X).squeeze()           # shape: (N,)  — raw scores
    loss   = loss_fn(logits, y.float())
    loss.backward()
    optimiser.step()

# Compute accuracy
with torch.no_grad():
    probs    = torch.sigmoid(model(X).squeeze())
    preds    = (probs > 0.5).long()
    accuracy = (preds == y).float().mean()
    print(f'Accuracy: {accuracy:.2%}')   # should be ~99%

# Inspect the learned decision boundary
w = model.weight.data.squeeze()   # shape: (2,)
b = model.bias.data.item()
print(f'w = {w.numpy()},  b = {b:.3f}')
# The decision boundary is the line:  w[0]*x1 + w[1]*x2 + b = 0

# Plot
x_plot = torch.linspace(-4, 4, 100)
y_plot = - (w[0] * x_plot + b) / w[1]
plt.plot(X0[:, 0], X0[:, 1], 'o', label='Class 0')
plt.plot(X1[:, 0], X1[:, 1], 'o', label='Class 1')
plt.plot(x_plot, y_plot, 'k', label='Decision boundary')
plt.legend()
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── Load MNIST and flatten to vectors ─────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),   # MNIST mean & std
    transforms.Lambda(lambda x: x.view(-1)),      # flatten 28x28 → 784
])

train_ds = datasets.MNIST('.', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('.', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512)

# ── Softmax classifier (logistic regression for 10 classes) ───────────
# nn.Linear maps each 784-dim image to 10 class scores
model     = nn.Linear(784, 10)              # 784*10 + 10 = 7,850 parameters
loss_fn   = nn.CrossEntropyLoss()           # softmax + cross-entropy
optimiser = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(5):
    model.train()
    for X_batch, y_batch in train_loader:
        optimiser.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimiser.step()

    # Validation accuracy
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            correct += (model(X_batch).argmax(1) == y_batch).sum().item()
    print(f'Epoch {epoch+1}  test accuracy: {correct/len(test_ds):.2%}')

# Typically reaches ~92% in 5 epochs — not bad for a linear model!

In [ ]:
import torch
import torch.nn as nn

# ── L2 regularisation in PyTorch ──────────────────────────────────────

# Option 1: Pass weight_decay to the optimiser (most common, cleanest)
model     = nn.Linear(784, 10)
optimiser = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4    # λ  — applied as L2 penalty on all parameters
)

# Option 2: Manual L2 penalty added to the loss (more transparent)
loss_fn = nn.CrossEntropyLoss()
lambda_ = 1e-4

X_batch = torch.randn(32, 784)
y_batch = torch.randint(0, 10, (32,))

logits    = model(X_batch)
data_loss = loss_fn(logits, y_batch)
l2_penalty = sum(p.pow(2).sum() for p in model.parameters())
total_loss_l2 = data_loss + lambda_ * l2_penalty
print(f'Data loss: {data_loss:.4f} ')
print(f'L2 penalty: {(lambda_*l2_penalty):.4f}')

# ── L1 regularisation (manual — no built-in optimiser shortcut) ───────
l1_penalty = sum(p.abs().sum() for p in model.parameters())
total_loss_l1 = data_loss + lambda_ * l1_penalty
print(f'L1 penalty: {lambda_*l1_penalty:.4f}')

# Insert training loop here where you use either total_loss_l2 or total_loss_l1

In [ ]:
import torch
import torch.nn as nn

# ── Softmax regression: the forward pass step by step ─────────────────
batch_size, n_features, n_classes = 4, 784, 10

W     = torch.randn(n_classes, n_features, requires_grad=True)
b     = torch.zeros(n_classes, requires_grad=True)
x     = torch.randn(batch_size, n_features)
y     = torch.tensor([0, 2, 1, 0])   # true class labels

# Step 1: Compute logits (class scores)
logits = x @ W.T + b               # shape: (batch, n_classes)

# Step 2: Softmax converts logits to probabilities
probs = torch.softmax(logits, dim=1)
print('Probabilities (should sum to 1 per row):')
print(probs.detach().round(decimals=3))
print('Row sums:', probs.sum(dim=1).detach())

# Step 3: Cross-entropy loss
# nn.CrossEntropyLoss = log_softmax + NLLLoss — takes raw logits, not probs
loss_fn = nn.CrossEntropyLoss()
loss    = loss_fn(logits, y)
print(f'\nCross-entropy loss: {loss.item():.4f}')

# Step 4: Backward pass
loss.backward()
print(f'Gradient of W: shape {W.grad.shape}')   # same as W

# ── Using nn.Linear (cleaner implementation) ──────────────────────────
model = nn.Sequential(
    nn.Linear(n_features, n_classes),   # W and b handled automatically
)
# nn.CrossEntropyLoss expects LOGITS (not softmax output)
loss2 = nn.CrossEntropyLoss()(model(x), y)

In [ ]:
import torch
import torch.nn as nn

# ── Visualising the limits of a linear classifier ─────────────────────
# XOR problem: NOT linearly separable
# Class 0: (0,0) and (1,1)   Class 1: (0,1) and (1,0)
X = torch.tensor([[0.,0.],[1.,1.],[0.,1.],[1.,0.]])
y = torch.tensor([0, 0, 1, 1])
print("Ground truth labels: ", y.tolist())

# Linear model (logistic regression)
lin_model = nn.Linear(2, 2)
opt       = torch.optim.Adam(lin_model.parameters(), lr=0.1)
loss_fn   = nn.CrossEntropyLoss()

for _ in range(1000):
    opt.zero_grad()
    loss_fn(lin_model(X), y).backward()
    opt.step()

preds_lin = lin_model(X).argmax(1)
print('Linear model predictions:', preds_lin.tolist())  # will fail on XOR

# Non-linear model (neural network with one hidden layer with ReLU)
mlp = nn.Sequential(nn.Linear(2,8), nn.ReLU(), nn.Linear(8,2))
opt2 = torch.optim.Adam(mlp.parameters(), lr=0.05)

for _ in range(2000):
    opt2.zero_grad()
    loss_fn(mlp(X), y).backward()
    opt2.step()

preds_mlp = mlp(X).argmax(1)
print('MLP predictions:', preds_mlp.tolist())   # correctly classifies XOR


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ── Toy data ──────────────────────────────────────────────────────────

np.random.seed(0)

X_train = np.random.randn(200, 2)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).astype(int)

X_test = np.random.randn(20, 2)
y_test = (X_test[:, 0] + X_test[:, 1] > 0).astype(int)

# Plot
plt.plot(X_train[y_train==0, 0], X_train[y_train==0, 1], 'o', label='Class 0')
plt.plot(X_train[y_train==1, 0], X_train[y_train==1, 1], 'o', label='Class 1')
plt.legend()
plt.show()

# ── K-Nearest Neighbours ──────────────────────────────────────────────

for k in [1, 5, 15]:

    model = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean'
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    print(f'K={k:2d}  accuracy: {accuracy:.0%}')

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans

# ── Demo: cluster three Gaussian blobs ────────────────────────────────

np.random.seed(42)

blobs = np.concatenate([
    np.random.randn(50, 2) + np.array([-3.,  0.]),
    np.random.randn(50, 2) + np.array([ 3.,  0.]),
    np.random.randn(50, 2) + np.array([ 0.,  3.]),
])

# ── K-means clustering ────────────────────────────────────────────────

model = KMeans(
    n_clusters=3,
    random_state=0,
    n_init=10
)

assignments = model.fit_predict(blobs)
centroids = model.cluster_centers_

# ── Inspect result ────────────────────────────────────────────────────

for k in range(3):
    count = np.sum(assignments == k)
    print(
        f'Cluster {k}: {count} points, '
        f'centroid ≈ {centroids[k].tolist()}'
    )

# ── Plot ──────────────────────────────────────────────────────────────

plt.figure()

plt.scatter(
    blobs[:, 0],
    blobs[:, 1],
    c=assignments
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker='X',
    s=150
)

plt.show()